In [1]:
# Read the file
with open("WikiArticle.txt", "r") as f:
    content = f.read()


In [2]:
# Split by the separator
raw_chunks = content.split("---")

# Clean up: remove empty chunks and strip whitespace
raw_chunks = [chunk.strip() for chunk in raw_chunks if chunk.strip()]

# Create the dictionary with IDs
chunks = {}
for i, chunk_text in enumerate(raw_chunks, start=1):
    doc_id = f"doc_{i:03d}"
    chunks[doc_id] = chunk_text

In [3]:
# Print the result to check
for doc_id, text in chunks.items():
    print(f"{doc_id}: {text[:100]}...")
    print()

doc_001: Dancing plague of 1518 - short summary
The dancing plague of 1518, or dance epidemic of 1518 (French...

doc_002: History Events
The outbreak began in July 1518 when a woman known as Frau Troffea and her daughter F...

doc_003: Veracity of deaths
Disagreement exists over whether people ultimately danced to death. Some sources ...

doc_004: Modern theories
Food poisoning
Some believe the dancing could have been brought on by food poisoning...

doc_005: Pop culture and media
The event inspired Jonathan Glazer's 2020 short film Strasbourg 1518.

It was ...



In [4]:
len(chunks)

5

In [5]:
for doc_id in chunks.keys():
    print(doc_id)

doc_001
doc_002
doc_003
doc_004
doc_005


In [6]:
print(chunks["doc_001"])

Dancing plague of 1518
The dancing plague of 1518, or dance epidemic of 1518 (French: Épidémie dansante de 1518; German: Straßburger Tanzwut), was a case of dancing mania that occurred in Strasbourg, Alsace (modern-day France), in the Holy Roman Empire from July 1518 to September 1518. Somewhere between 50 and 400 people took to dancing for weeks. There are many theories behind the phenomenon, the most popular being stress-induced mass hysteria, suggested by John Waller. Other theories include ergot poisoning. There is controversy concerning the number of deaths, but the total is unknown.


In [7]:
import pandas as pd

df = pd.read_csv("ground_truth.csv")
df

,qa_id,doc_id,question,answer
0,qa_001,doc_001,What is the dancing plague of 1518?,"The dancing plague of 1518, or dance epidemic ..."
1,qa_002,doc_001,How many people were affected by the dancing p...,Somewhere between 50 and 400 people took to da...
2,qa_003,doc_002,How did the outbreak of the dancing plague in ...,The outbreak began in July 1518 when a woman k...
3,qa_004,doc_002,What cured the dancing plague of 1518?,"No one knew what caused this reaction, which m..."
4,qa_005,doc_003,How many people died during the dancing plague...,Some sources claim that for a period the plagu...
5,qa_006,doc_003,Did people really danced till they were dead?,Disagreement exists over whether people ultima...
6,qa_007,doc_004,What are the modern theories about the cause o...,There are modern theories that say the dancing...
7,qa_008,doc_004,Was the dancing plague a psychological illness?,It could have been an example of fully develop...
8,qa_009,doc_005,Are there any fictional books about the Dancin...,The book series A Collection of Utter Speculat...
9,qa_010,doc_005,Are there any films or movies made about the D...,The event inspired Jonathan Glazer's 2020 shor...


In [8]:
ground_truth = {}

for _, row in df.iterrows():
    qa_id = row["qa_id"]
    ground_truth[qa_id] = {
        "doc_id": row["doc_id"],
        "question": row["question"],
        "answer": row["answer"]
    }

# Check your work
for qa_id, data in ground_truth.items():
    print(f"{qa_id}: {data['question']} (from {data['doc_id']})")

qa_001: What is the dancing plague of 1518? (from doc_001)
qa_002: How many people were affected by the dancing plague of 1518? (from doc_001)
qa_003: How did the outbreak of the dancing plague in 1518 start? (from doc_002)
qa_004: What cured the dancing plague of 1518? (from doc_002)
qa_005: How many people died during the dancing plague of 1518? (from doc_003)
qa_006: Did people really danced till they were dead? (from doc_003)
qa_007: What are the modern theories about the cause of the dancing plage of 1518? (from doc_004)
qa_008: Was the dancing plague a psychological illness? (from doc_004)
qa_009: Are there any fictional books about the Dancing Plague? (from doc_005)
qa_010: Are there any films or movies made about the Dancing Plague? (from doc_005)


In [18]:
# Load your CSV
df = pd.read_csv("ground_truth.csv")

# Create the list of dictionaries with chunks
ground_truth_list = []

for qa_id, data in ground_truth.items():
    record = {
        "qa_id": qa_id,
        "doc_id": data["doc_id"],
        "question": data["question"],
        "answer": data["answer"]
    }
    ground_truth_list.append(record)

In [ ]:
ground_truth_list = []

for qa_id, data in ground_truth.items():
    doc_id = data["doc_id"]
    
    # Get the chunk text using the doc_id
    chunk_text = chunks[doc_id]
    
    record = {
        "qa_id": qa_id,
        "doc_id": doc_id,
        "chunk": chunk_text,
        "question": data["question"],
        "answer": data["answer"]
    }
    ground_truth_list.append(record)

In [21]:
ground_truth_list[0]

{'qa_id': 'qa_001',
 'doc_id': 'doc_001',
 'chunk': 'Dancing plague of 1518\nThe dancing plague of 1518, or dance epidemic of 1518 (French: Épidémie dansante de 1518; German: Straßburger Tanzwut), was a case of dancing mania that occurred in Strasbourg, Alsace (modern-day France), in the Holy Roman Empire from July 1518 to September 1518. Somewhere between 50 and 400 people took to dancing for weeks. There are many theories behind the phenomenon, the most popular being stress-induced mass hysteria, suggested by John Waller. Other theories include ergot poisoning. There is controversy concerning the number of deaths, but the total is unknown.',
 'question': 'What is the dancing plague of 1518?',
 'answer': 'The dancing plague of 1518, or dance epidemic of 1518, was a case of dancing mania that occurred in Strasbourg'}

In [22]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "question", "answer"],
)

index.fit(ground_truth_list)

In [28]:
question = "What is the dancing plague?"

search_results = index.search(
    question,
    boost_dict={"chunk": 1.0, "question": 2.0, "answer": 1.0},
    num_results=5
)

search_results

[{'qa_id': 'qa_001',
  'doc_id': 'doc_001',
  'chunk': 'Dancing plague of 1518\nThe dancing plague of 1518, or dance epidemic of 1518 (French: Épidémie dansante de 1518; German: Straßburger Tanzwut), was a case of dancing mania that occurred in Strasbourg, Alsace (modern-day France), in the Holy Roman Empire from July 1518 to September 1518. Somewhere between 50 and 400 people took to dancing for weeks. There are many theories behind the phenomenon, the most popular being stress-induced mass hysteria, suggested by John Waller. Other theories include ergot poisoning. There is controversy concerning the number of deaths, but the total is unknown.',
  'question': 'What is the dancing plague of 1518?',
  'answer': 'The dancing plague of 1518, or dance epidemic of 1518, was a case of dancing mania that occurred in Strasbourg'},
 {'qa_id': 'qa_004',
  'doc_id': 'doc_002',
  'chunk': 'History\nEvents\nThe outbreak began in July 1518 when a woman known as Frau Troffea and her daughter Fräulein

In [26]:
[doc["question"] for doc in search_results]

['What is the dancing plague of 1518?',
 'What cured the dancing plague of 1518?',
 'What are the modern theories about the cause of the dancing plage of 1518?',
 'Are there any fictional books about the Dancing Plague?',
 'How many people died during the dancing plague of 1518?']

In [ ]:
def search(question"):
    boost_dict = {"chunk": 1.0, "question": 2.0, "answer": 1.0}

    return index.search(
        question,
        boost_dict=boost_dict,
        num_results=5
    )

In [29]:
print("Hello World")

Hello World


In [4]:
import pandas as pd

# Read the file
with open("WikiArticle.txt", "r") as f:
    content = f.read()

# Split by the separator
raw_chunks = content.split("---")

# Clean up: remove empty chunks and strip whitespace
raw_chunks = [chunk.strip() for chunk in raw_chunks if chunk.strip()]

# Create the dictionary with IDs
chunks = {}
for i, chunk_text in enumerate(raw_chunks, start=1):
    doc_id = f"doc_{i:03d}"
    chunks[doc_id] = chunk_text

# Process chunks into DataFrame
data = []
for doc_id, text in chunks.items():
    lines = text.split('\n')
    section = lines[0].strip()  # First line is the section title
    chunk_content = '\n'.join(lines[1:]).strip()  # Everything else is the content
    data.append({
        'doc_id': doc_id,
        'section': section,
        'chunk': chunk_content
    })

# Create DataFrame and save
df = pd.DataFrame(data)
df.to_csv('dancing_plague_chunks.csv', index=False)

print("✅ CSV saved as 'dancing_plague_chunks.csv'")
print("\nPreview:")
print(df)

✅ CSV saved as 'dancing_plague_chunks.csv'

Preview:
    doc_id                                 section  \
0  doc_001  Dancing plague of 1518 - short summary   
1  doc_002                          History Events   
2  doc_003                      Veracity of deaths   
3  doc_004                         Modern theories   
4  doc_005                   Pop culture and media   

                                               chunk  
0  The dancing plague of 1518, or dance epidemic ...  
1  The outbreak began in July 1518 when a woman k...  
2  Disagreement exists over whether people ultima...  
3  Food poisoning\nSome believe the dancing could...  
4  The event inspired Jonathan Glazer's 2020 shor...  
